In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\scripts')
from _offline import enforce_offline as __guard, blocked_attempts as __blocked
__guard()
import capture_runresults_213 as __capture
__capture.install()
import agentic_systems as __harness_toolkit
assert __harness_toolkit.__version__ == "2.1.3"
assert "site-packages" in __harness_Path(__harness_toolkit.__file__).parts

# Agent: secuencia, paralelo y decisiones

El propietario cambia su flujo; conserva el mismo run y el mismo RunResult.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as toolkit


In [3]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

def choose_next(data):
    return "multiply" if data["value"] >= 0 else "END"

def combine(branches):
    return {"value": sum(row["value"] for row in branches.values())}

In [4]:
for backend in ("native", "langgraph"):
    sequential = toolkit.Agent("sequential", toolkit.Sequence((increase, multiply)), skills=(arithmetic,), backend=backend)
    parallel = toolkit.Agent("parallel", toolkit.Parallel((increase, multiply), join=combine), skills=(arithmetic,), backend=backend)
    decision = toolkit.Agent("decision", toolkit.Graph(
        start="increase", nodes=(("increase", increase), ("multiply", multiply)),
        edges=(("increase", choose_next), ("multiply", "END")),
    ), skills=(arithmetic,), backend=backend)
    for agent, expected in ((sequential, 8), (parallel, 10), (decision, 8)):
        result = agent.run({"value": 3})
        assert result.ok and result.data == {"value": expected}
        print(backend, agent.name, {"input": 3, "output": result.data})
    assert decision.run({"value": -3}).data == {"value": -2}

native sequential {'input': 3, 'output': {'value': 8}}
native parallel {'input': 3, 'output': {'value': 10}}
native decision {'input': 3, 'output': {'value': 8}}


langgraph sequential {'input': 3, 'output': {'value': 8}}
langgraph parallel {'input': 3, 'output': {'value': 10}}
langgraph decision {'input': 3, 'output': {'value': 8}}


Graph también admite ciclos limitados. La condición es una función normal; max_visits evita un ciclo ilimitado. El límite y las excepciones conservan los resultados anteriores.

In [5]:
loop = toolkit.Agent('loop', toolkit.Graph(nodes=(('inc', increase),), edges=(('inc', lambda d: 'inc' if d['value'] < 3 else 'END'),), start='inc', max_visits=4), skills=(arithmetic,))
assert loop.run({'value':0}).data == {'value':3}
print(loop.inspect())

{'name': 'loop', 'kind': 'agent', 'backend': 'native', 'native_type': None, 'children': [{'name': 'graph', 'kind': 'graph', 'backend': 'native', 'native_type': None, 'children': [{'name': 'inc', 'kind': 'action', 'backend': 'python', 'native_type': None, 'children': []}]}]}


Secuencia: 3 → 4 → 8. Paralelo: el mismo 3 alimenta dos ramas → 4 y 6 → join suma 10. Decisión: 3 → 4 → multiply → 8; −3 → −2 → END. La condición recibe la salida del nodo anterior. El backend LangGraph construye un StateGraph real; no cambia el proveedor de agentes anidados.

In [6]:
__capture.save('C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\revisions\\2.1.3\\closeout-evidence\\notebook-runresults\\04-02_agent_flows.json', 'tutorials/02_agent_flows.ipynb')
assert __blocked() == 0